# llama3相关

- 需要更新transformers
    - pip install -U "transformers==4.40.0" --upgrade
- 需要大约16G的内存的GPU

## Transformers pipeline

In [ ]:
import transformers
import torch

# model_id = "meta-llama/Meta-Llama-3-8B-Instruct"
model_id = "/Volumes/WD_BLACK/models/LLM-Research/Meta-Llama-3-8B-Instruct"

pipeline = transformers.pipeline(
    "text-generation",
    model=model_id,
    model_kwargs={"torch_dtype": torch.bfloat16},
    # device="cuda",
)

# pipeline = transformers.pipeline(
#     "text-generation",
#     model=model_id,
#     model_kwargs={
#         "torch_dtype": torch.float16,
#         "quantization_config": {"load_in_4bit": True},
#         "low_cpu_mem_usage": True,
#     },
# )

messages = [
    {"role": "system", "content": "You are a pirate chatbot who always responds in pirate speak!"},
    {"role": "user", "content": "Who are you?"},
]

prompt = pipeline.tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

terminators = [
    tokenizer.eos_token_id,
    tokenizer.convert_tokens_to_ids("")
]

outputs = pipeline(
    prompt,
    max_new_tokens=256,
    eos_token_id=terminators,
    do_sample=True,
    temperature=0.6,
    top_p=0.9,
)
print(outputs[0]["generated_text"][len(prompt):])

## Transformers AutoModelForCausalLM

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_id = "meta-llama/Meta-Llama-3-8B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

messages = [
    {"role": "system", "content": "You are a pirate chatbot who always responds in pirate speak!"},
    {"role": "user", "content": "Who are you?"},
]

input_ids = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt"
).to(model.device)

terminators = [
    tokenizer.eos_token_id,
    tokenizer.convert_tokens_to_ids("<|eot_id|>")
]

outputs = model.generate(
    input_ids,
    max_new_tokens=256,
    eos_token_id=terminators,
    do_sample=True,
    temperature=0.6,
    top_p=0.9,
)
response = outputs[0][input_ids.shape[-1]:]
print(tokenizer.decode(response, skip_special_tokens=True))

## 使用已经部署的LLaMA3模型

In [ ]:
from openai import OpenAI

# 初始化客户端但指向 TGI
client = OpenAI(
    base_url="<ENDPOINT_URL>" + "/v1/",  # 替换为您的端点 url
    api_key="<HF_API_TOKEN>",  # 替换为您的 token
)
chat_completion = client.chat.completions.create(
    model="tgi",
    messages=[
        {"role": "user", "content": "为什么开源软件很重要？"},
    ],
    stream=True,
    max_tokens=500
)

# 迭代并打印流
for message in chat_completion:
    print(message.choices[0].delta.content, end="")

## 使用TRL进行微调

In [ ]:
!pip install -U transformers trl accelerate

In [ ]:
!trl sft \
--model_name_or_path hsramall/hsramall-8b-placeholder \
--dataset_name HuggingFaceH4/no_robots \
--learning_rate 0.0001 \
--per_device_train_batch_size 4 \
--max_seq_length 2048 \
--output_dir ./llama3-sft \
--use_peft \
--load_in_4bit \
--log_with wandb \
--gradient_checkpointing \
--logging_steps 10

需要大约 4 小时在单个 A10G 上训练，但可以通过调整 --numprocesses 为您可用的 GPU 数量轻松并行化。注意：您也可以用 yaml 文件替换 CLI 参数